# Analysis

Tables and plots from `results/`. Run `caris evaluate` and `caris sweep` first.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from caris.evaluate import paired_difference

RESULTS = Path("results") if Path("results").is_dir() else Path("../results")
results = {path.stem: json.loads(path.read_text(encoding="utf-8")) for path in sorted(RESULTS.glob("*.json"))}
print(f"{len(results)} results files")

## Comparison table

One row per results file: per-image means, pooled precision and recall, the recall ceiling, list length and tagging time.

In [ ]:
def row(result):
    experiment, summary = result["experiment"], result["summary"]
    return {
        "split": result["split"],
        "VLM": experiment["tagging"]["model"],
        "embedding": experiment["embedding"]["model"],
        "code text": experiment["embedding"]["text"],
        "images": result["images"]["evaluated"],
        **{name: summary[name]["mean"] for name in ("hit", "precision", "recall", "f1", "h_f1")},
        "pooled precision": summary["pooled"]["precision"],
        "pooled recall": summary["pooled"]["recall"],
        "recall ceiling": summary["ceiling"]["recall"],
        "codes per image": summary["codes_per_image"],
        "seconds per image": summary["median_seconds_per_image"],
    }


pd.DataFrame({name: row(result) for name, result in results.items() if result["summary"]}).T

## Precision and recall: top-k against thresholds

Each `caris sweep` table holds three kinds of rule: top-k only, a threshold only, and a threshold with a top-k cap.

In [ ]:
CAPS = (None, 10, 20)  # which threshold curves to draw: uncapped, and capped at these k

fig, ax = plt.subplots(figsize=(7, 5))
for path in sorted(RESULTS.glob("*__sweep.csv")):
    sweep = pd.read_csv(path)
    name = path.stem.removesuffix("__sweep")
    top_k_only = sweep[sweep["threshold"].isna()].sort_values("top_k")
    ax.plot(top_k_only["pooled_recall"], top_k_only["pooled_precision"], marker="o", label=f"{name}: top-k")
    for cap in CAPS:
        capped = sweep["top_k"].isna() if cap is None else sweep["top_k"] == cap
        curve = sweep[sweep["threshold"].notna() & capped]
        label = "threshold" if cap is None else f"threshold, top {cap}"
        ax.plot(curve["pooled_recall"], curve["pooled_precision"], marker=".", label=f"{name}: {label}")
ax.set(xlabel="recall (pooled)", ylabel="precision (pooled)")
ax.legend(fontsize="small")
plt.show()

## By branch

Pooled recall per top-level Iconclass branch, for the stratified analysis.

In [ ]:
pd.DataFrame({
    name: {branch: values["recall"] for branch, values in result["summary"]["by_branch"].items()}
    for name, result in results.items() if result["summary"]
})

## Paired comparison

Whether one results file beats another on the images both evaluated. If the 95% interval spans 0, the difference isn't clear.

In [ ]:
first, second = list(results)[:2]  # replace with the two results files to compare
for metric in ("hit", "recall", "h_f1"):
    print(metric, paired_difference(results[first], results[second], metric))